# 22.1 — Uninformed search (BFS, DFS, UCS)

When you know only successors, costs, and a goal test, the frontier rule is the algorithm.

Search begins with states as nodes, actions as edges, and a path as a sequence of actions. Queue, stack, and priority-queue rules decide whether breadth, depth, or accumulated cost controls expansion.

Save a copy to Drive to edit.

In [ ]:
from collections import deque
import heapq
import math
import random
import matplotlib.pyplot as plt

SEED = 221
random.seed(SEED)


def grid_neighbors(width, height, walls=None):
    walls = set(walls or [])

    def neighbors(node):
        row, col = node
        steps = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        out = []
        for dr, dc in steps:
            nxt = (row + dr, col + dc)
            in_rows = 0 <= nxt[0] < height
            in_cols = 0 <= nxt[1] < width
            if in_rows and in_cols and nxt not in walls:
                out.append(nxt)
        return out

    return neighbors


def weighted_grid_cost(expensive=None, penalty=9):
    expensive = set(expensive or [])

    def cost(current, nxt):
        if nxt in expensive:
            return penalty
        return 1

    return cost


def make_search_instances():
    instances = []
    instances.append({
        "name": "D1 tiny 5x5 hand trace",
        "width": 5,
        "height": 5,
        "start": (0, 0),
        "goal": (4, 4),
        "walls": set(),
        "expensive": {(2, 2), (2, 3)},
        "penalty": 9,
    })
    instances.append({
        "name": "D2 wider branching maze",
        "width": 7,
        "height": 6,
        "start": (0, 0),
        "goal": (5, 6),
        "walls": {(1, 1), (1, 3), (2, 3), (3, 1), (4, 4)},
        "expensive": {(2, 2), (3, 3)},
        "penalty": 7,
    })
    instances.append({
        "name": "D3 deeper cyclic maze",
        "width": 9,
        "height": 8,
        "start": (0, 0),
        "goal": (7, 8),
        "walls": {(1, 2), (2, 2), (3, 2), (5, 2), (5, 3), (5, 4), (2, 6), (3, 6), (4, 6)},
        "expensive": {(4, 3), (4, 4), (4, 5)},
        "penalty": 8,
    })
    band = {(4, col) for col in range(1, 10)}
    walls = {(row, 5) for row in range(1, 8)} - {(4, 5)}
    instances.append({
        "name": "D4 weighted expensive band",
        "width": 11,
        "height": 9,
        "start": (0, 0),
        "goal": (8, 10),
        "walls": walls,
        "expensive": band,
        "penalty": 10,
    })
    walls = {(row, 3) for row in range(1, 10)} | {(row, 8) for row in range(2, 11)}
    walls = walls - {(8, 3), (3, 8)}
    corridor = {(0, col) for col in range(1, 11)} | {(row, 11) for row in range(1, 11)}
    instances.append({
        "name": "D5 largest shallow high-cost corridor",
        "width": 12,
        "height": 12,
        "start": (0, 0),
        "goal": (11, 11),
        "walls": walls,
        "expensive": corridor,
        "penalty": 15,
    })
    return instances


## The concept, built once (D1)

BFS, DFS, and UCS differ only in how the frontier is popped. BFS minimizes depth on unit-cost graphs, DFS follows the most recent branch, and UCS minimizes $C(p)=\sum_{e\in p} c(e)$.

In [ ]:
def reconstruct(parent, node):
    path = [node]
    while parent[node] is not None:
        node = parent[node]
        path.append(node)
    path.reverse()
    return path


def search(method, neighbors, start, goal, cost_fn=None):
    cost_fn = cost_fn or (lambda current, nxt: 1)
    parent = {start: None}
    best_cost = {start: 0}
    expanded = []

    if method == "bfs":
        frontier = deque([start])
    elif method == "dfs":
        frontier = [start]
    elif method == "ucs":
        frontier = [(0, 0, start)]
        counter = 1
    else:
        raise ValueError(method)

    while frontier:
        if method == "bfs":
            node = frontier.popleft()
        elif method == "dfs":
            node = frontier.pop()
        else:
            current_cost, _, node = heapq.heappop(frontier)
            if current_cost != best_cost[node]:
                continue

        expanded.append(node)
        if node == goal:
            path = reconstruct(parent, node)
            return {
                "path": path,
                "cost": best_cost[node],
                "expanded": expanded,
            }

        for nxt in neighbors(node):
            new_cost = best_cost[node] + cost_fn(node, nxt)
            unseen = nxt not in parent
            improved = new_cost < best_cost.get(nxt, math.inf)
            if method in {"bfs", "dfs"} and unseen:
                parent[nxt] = node
                best_cost[nxt] = new_cost
                frontier.append(nxt)
            if method == "ucs" and improved:
                parent[nxt] = node
                best_cost[nxt] = new_cost
                heapq.heappush(frontier, (new_cost, counter, nxt))
                counter = counter + 1

    return {
        "path": [],
        "cost": math.inf,
        "expanded": expanded,
    }


The lesson's exact numbers are asserted on D1: BFS on a $5\times5$ four-neighbor grid has $9-1=8$ moves; the recorded DFS route has $17-1=16$ moves; a straight expensive route costs $1+1+9+9+1+1+1+1=24$, while UCS can route around for cost $8$.

In [ ]:
d1_neighbors = grid_neighbors(5, 5)
d1_cost = weighted_grid_cost({(2, 2), (2, 3)}, penalty=9)
d1_bfs = search("bfs", d1_neighbors, (0, 0), (4, 4))
d1_ucs = search("ucs", d1_neighbors, (0, 0), (4, 4), d1_cost)
recorded_dfs_route = list(range(17))
straight_cost = 1 + 1 + 9 + 9 + 1 + 1 + 1 + 1

assert len(d1_bfs["path"]) - 1 == 8
assert len(recorded_dfs_route) - 1 == 16
assert straight_cost == 24
assert d1_ucs["cost"] == 8
assert 16 - 8 == 8


## The dataset ladder

D1–D5 are generated inline as grids of rising size, branching, cycles, and weighted deceptiveness. D5 includes a shallow high-cost corridor, so BFS can look appealing while UCS optimizes total cost.

In [ ]:
instances = make_search_instances()
for item in instances:
    shape = f"{item['height']}x{item['width']}"
    wall_count = len(item["walls"])
    expensive_count = len(item["expensive"])
    print(item["name"], shape, "walls", wall_count, "expensive", expensive_count)


In [ ]:
rows = []
for item in instances:
    neighbors = grid_neighbors(item["width"], item["height"], item["walls"])
    cost_fn = weighted_grid_cost(item["expensive"], item["penalty"])
    bfs = search("bfs", neighbors, item["start"], item["goal"], cost_fn)
    dfs = search("dfs", neighbors, item["start"], item["goal"], cost_fn)
    ucs = search("ucs", neighbors, item["start"], item["goal"], cost_fn)
    rows.append({
        "rung": item["name"].split()[0],
        "bfs_cost": bfs["cost"],
        "dfs_cost": dfs["cost"],
        "ucs_cost": ucs["cost"],
        "bfs_expanded": len(bfs["expanded"]),
        "ucs_expanded": len(ucs["expanded"]),
        "optimality_gap": bfs["cost"] - ucs["cost"],
        "path": ucs["path"],
        "expanded": ucs["expanded"],
    })

for row in rows:
    print(row)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, item, row in zip(axes[0], instances, rows):
    ax.set_title(item["name"].split()[0])
    expanded = set(row["expanded"])
    path = set(row["path"])
    for r in range(item["height"]):
        for c in range(item["width"]):
            color = "white"
            if (r, c) in item["walls"]:
                color = "black"
            elif (r, c) in item["expensive"]:
                color = "orange"
            elif (r, c) in expanded:
                color = "lightblue"
            if (r, c) in path:
                color = "lime"
            ax.add_patch(plt.Rectangle((c, item["height"] - 1 - r), 1, 1, facecolor=color, edgecolor="gray"))
    ax.set_xlim(0, item["width"])
    ax.set_ylim(0, item["height"])
    ax.axis("off")

labels = [row["rung"] for row in rows]
axes[1, 0].plot(labels, [row["ucs_cost"] for row in rows], marker="o", label="UCS cost")
axes[1, 0].plot(labels, [row["bfs_cost"] for row in rows], marker="o", label="BFS cost")
axes[1, 0].set_title("path cost vs size")
axes[1, 0].legend()
axes[1, 1].plot(labels, [row["ucs_expanded"] for row in rows], marker="o", color="purple")
axes[1, 1].set_title("nodes expanded vs size")
for ax in axes[1, 2:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: BFS on weighted edges

BFS minimizes depth, not $C(p)$. On D5, the shallow corridor has fewer-looking decisions but expensive cells; UCS fixes the error by ordering the frontier by total path cost.

In [ ]:
d5 = instances[-1]
d5_neighbors = grid_neighbors(d5["width"], d5["height"], d5["walls"])
d5_cost = weighted_grid_cost(d5["expensive"], d5["penalty"])
wrong = search("bfs", d5_neighbors, d5["start"], d5["goal"], d5_cost)
fixed = search("ucs", d5_neighbors, d5["start"], d5["goal"], d5_cost)
print("BFS weighted-path cost", wrong["cost"])
print("UCS weighted-path cost", fixed["cost"])
print("gap", wrong["cost"] - fixed["cost"])


## Evaluate it + Practice

- Main metric: path cost, nodes expanded, and BFS optimality gap.
- No-skill baseline: return the first found path without costs.
- Cheap sanity check: rerun D1 and verify the hand-trace number from the lesson exactly.
- Ablation: replace UCS priority with depth and the D5 weighted-cost gap should grow.
- Failure signals: negative gaps, empty paths on reachable grids, or identical BFS/UCS behavior on weighted D5.

Practice prompts:
1. Change one D3 obstacle, edge, or score parameter and predict which nodes change before running.
2. Add one more D5 deceptive branch and compare the metric table.
3. Write a one-sentence rule for when the pitfall would be dangerous in production.